# Notebook 09 — Recomendador Item-Item (Similitud Coseno)

En los notebooks anteriores construimos dos formas distintas de recomendar productos:

- **NB06 (Popularidad):** el piso — recomienda lo mismo a todo el mundo.
- **NB07/08 (Market Basket Analysis):** reglas de asociación del tipo "si comprás A y B, solés comprar C", basadas en **co-ocurrencia dentro del mismo pedido**.

Acá construimos un tercer enfoque, distinto en su lógica: **filtrado colaborativo item-item basado en similitud coseno**.

**La idea central:** en vez de mirar qué productos aparecen juntos en un mismo pedido (como el MBA), miramos qué productos son comprados por **los mismos usuarios a lo largo del tiempo** (no necesariamente en el mismo pedido). Dos productos son "similares" si tienden a ser comprados por el mismo conjunto de gente. Por ejemplo: leche de almendras y pan sin gluten pueden no estar nunca en el mismo pedido, pero si los compran mayormente los mismos usuarios (con perfil "alimentación especial"), el modelo los va a considerar similares.

**Qué vamos a hacer en esta notebook, en orden:**
1. Explicar la matriz usuario-producto y la similitud coseno con manos en la masa (no solo teoría).
2. Entrenar el modelo con el mismo split *leave-last-order-out* que usamos en NB06, para poder comparar métricas de forma justa.
3. Armar una función `recommend_from_cart` — igual en espíritu a la de NB07/08, pero basada en similitud en vez de reglas.
4. Probarla con el **mismo carrito de ejemplo** que usamos en NB07/08 (`Organic Baby Spinach` + `Organic Hass Avocado`), para poder comparar directamente qué recomienda cada enfoque.
5. Validar el modelo con **dos tipos de métricas complementarias**:
   - **(A) Métricas de tarea** (Precision@k, Recall@k, HitRate@k, MAP@k, Coverage): ¿el modelo predice bien lo que el usuario compró después? Comparables con NB06/07/08.
   - **(B) Coherencia de categoría** (métrica intrínseca, específica de este modelo): ¿los productos que el modelo considera "similares" tienen sentido de negocio (caen en el mismo pasillo)?

Cada celda de métrica va a explicar qué significa el número y cómo interpretarlo — el objetivo es que termines la notebook entendiendo *por qué* confiar (o no) en este modelo, no solo viendo que "corrió".


## Fase 0 — Setup e Imports

In [46]:
import sys
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

# ── Rutas ─────────────────────────────────────────────────────────────────────
DATA_RAW  = Path('../data/raw/instacart')
DATA_PROC = Path('../data/processed')
MODELS    = Path('../models')

_project_root = str(Path('..').resolve())
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

# ── Reproducibilidad ──────────────────────────────────────────────────────────
RANDOM_STATE = 42

# ── Parámetros del modelo ─────────────────────────────────────────────────────
MIN_USER_SUPPORT = 0.003   # un producto necesita que >=0.3% de los usuarios lo hayan comprado
                            # para entrar en la matriz de similitud (ver Fase 1)
MAX_CONSEQUENT_SUPPORT = 0.30  # tope de "no recomendar lo obvio" (ver seccion de politica)
TOP_N       = 10           # top-N para la demo interactiva
K_VALUES    = [5, 10, 20]  # valores de k para las metricas @k

print('Setup completo.')
print(f'  DATA_RAW  : {DATA_RAW.resolve()}')
print(f'  DATA_PROC : {DATA_PROC.resolve()}')
print(f'  MODELS    : {MODELS.resolve()}')


Setup completo.
  DATA_RAW  : C:\henry\ProyectoFinal-DataScience-Henry\data\raw\instacart
  DATA_PROC : C:\henry\ProyectoFinal-DataScience-Henry\data\processed
  MODELS    : C:\henry\ProyectoFinal-DataScience-Henry\models


## Carga de datos crudos

In [47]:
# Pedidos: id de orden, usuario, tipo de conjunto y numero de orden
orders = pd.read_csv(
    DATA_RAW / 'orders.csv',
    usecols=['order_id', 'user_id', 'eval_set', 'order_number'],
    dtype={'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16'},
)
orders_prior = orders[orders['eval_set'] == 'prior'].copy()

# Productos comprados en pedidos prior
op_prior = pd.read_csv(
    DATA_RAW / 'order_products__prior.csv',
    usecols=['order_id', 'product_id', 'reordered'],
    dtype={'order_id': 'int32', 'product_id': 'int32', 'reordered': 'int8'},
)

# Catalogo de productos (con pasillo y departamento, los vamos a usar en la
# metrica de coherencia de categoria - seccion B)
products = pd.read_csv(
    DATA_RAW / 'products.csv',
    usecols=['product_id', 'product_name', 'aisle_id', 'department_id'],
    dtype={'product_id': 'int32', 'aisle_id': 'int16', 'department_id': 'int16'},
)
aisles = pd.read_csv(DATA_RAW / 'aisles.csv', dtype={'aisle_id': 'int16'})

id_to_name = products.set_index('product_id')['product_name'].to_dict()
product_to_aisle = products.set_index('product_id')['aisle_id'].to_dict()
aisle_id_to_name = aisles.set_index('aisle_id')['aisle'].to_dict()

# Vista unificada: cada linea de compra con su usuario
op_user = op_prior.merge(
    orders_prior[['order_id', 'user_id', 'order_number']],
    on='order_id',
    how='inner',
)
op_user['user_id']      = op_user['user_id'].astype('int32')
op_user['order_number'] = op_user['order_number'].astype('int16')

print(f'orders_prior : {orders_prior.shape}  -- {orders_prior["user_id"].nunique():,} usuarios, {orders_prior["order_id"].nunique():,} pedidos')
print(f'op_prior     : {op_prior.shape}')
print(f'products     : {products.shape}  -- {products["product_id"].nunique():,} productos unicos')
print(f'op_user      : {op_user.shape}')


orders_prior : (3214874, 4)  -- 206,209 usuarios, 3,214,874 pedidos
op_prior     : (32434489, 3)
products     : (49688, 4)  -- 49,688 productos unicos
op_user      : (32434489, 5)


## 1. La matriz usuario-producto y la similitud coseno (teoria con ejemplo)

Antes de programar nada con el dataset completo, veamos la idea con 4 usuarios y 3 productos de juguete.

Imaginemos que armamos una tabla donde cada **fila es un usuario** y cada **columna es un producto**, con un `1` si ese usuario compró alguna vez ese producto (y `0` si no). Eso es la **matriz usuario-producto** (en inglés, *user-item matrix*):

|        | Banana | Leche de almendras | Pan sin gluten |
|--------|:------:|:-------------------:|:---------------:|
| user 1 |   1    |          1           |        1         |
| user 2 |   1    |          0           |        0         |
| user 3 |   0    |          1           |        1         |
| user 4 |   1    |          1           |        0         |

Cada **columna** de esta tabla es un vector que describe a un producto por *quién lo compra*. Por ejemplo, "Leche de almendras" = `[1, 0, 1, 1]` y "Pan sin gluten" = `[1, 0, 1, 0]`.

La **similitud coseno** entre dos vectores mide el ángulo entre ellos (no la magnitud). Para vectores binarios, esto tiene una lectura muy directa:

```
cos(A, B) = (A · B) / (||A|| * ||B||)
          = (usuarios que compraron A Y B) / raiz(usuarios que compraron A) * raiz(usuarios que compraron B)
```

- El **numerador** cuenta cuántos usuarios compraron *ambos* productos (el "solapamiento").
- El **denominador** normaliza por qué tan populares son A y B por separado.

Esto es clave: **si no normalizáramos, un producto ultra-popular (banana) parecería "similar" a casi todo**, solo porque casi todo el mundo lo compra, no porque haya una relación real. El coseno corrige eso — es matemáticamente parecido al índice de Jaccard, pero pesado distinto.

Rango del coseno: `0` (ningún usuario en común) a `1` (exactamente el mismo conjunto de compradores).

Calculemoslo a mano para nuestro ejemplo, para que quede el concepto antes de escalarlo a 200.000 usuarios reales.


In [48]:
# Ejemplo de juguete: 4 usuarios x 3 productos, matriz binaria
toy = pd.DataFrame(
    {
        'Banana':              [1, 1, 0, 1],
        'Leche_de_almendras':  [1, 0, 1, 1],
        'Pan_sin_gluten':      [1, 0, 1, 0],
    },
    index=['user1', 'user2', 'user3', 'user4'],
)
display(toy)

toy_sim = cosine_similarity(toy.T)  # transponemos: ahora cada FILA es un producto
toy_sim_df = pd.DataFrame(toy_sim, index=toy.columns, columns=toy.columns)

print('Matriz de similitud coseno entre productos (juguete):')
display(toy_sim_df.round(3))

print('\nLeyendo la tabla:')
print(f"  Leche_de_almendras vs Pan_sin_gluten : {toy_sim_df.loc['Leche_de_almendras', 'Pan_sin_gluten']:.3f}"
      f"  (los compran los mismos 2 usuarios: user1 y user3 -> similitud alta)")
print(f"  Banana vs Pan_sin_gluten             : {toy_sim_df.loc['Banana', 'Pan_sin_gluten']:.3f}"
      f"  (solo se solapan en user1 -> similitud mas baja)")


,Banana,Leche_de_almendras,Pan_sin_gluten
user1,1,1,1
user2,1,0,0
user3,0,1,1
user4,1,1,0


Matriz de similitud coseno entre productos (juguete):


,Banana,Leche_de_almendras,Pan_sin_gluten
Banana,1.000,0.667,0.408
Leche_de_almendras,0.667,1.000,0.816
Pan_sin_gluten,0.408,0.816,1.000



Leyendo la tabla:
  Leche_de_almendras vs Pan_sin_gluten : 0.816  (los compran los mismos 2 usuarios: user1 y user3 -> similitud alta)
  Banana vs Pan_sin_gluten             : 0.408  (solo se solapan en user1 -> similitud mas baja)


## 2. Split *leave-last-order-out* (mismo esquema que NB06)

Para poder comparar este modelo con Popularidad (NB06) y con el MBA (NB07/08) de forma justa, usamos exactamente el mismo esquema de validación:

- **Train:** todos los pedidos de cada usuario **excepto el último**.
- **Test:** el último pedido de cada usuario (el ground truth: lo que realmente compró después).

El modelo se entrena **solo con train** — nunca ve el último pedido de ningún usuario. Así evitamos *data leakage* (que el modelo "haga trampa" viendo el futuro).


In [49]:
# Ultimo order_number por usuario (el pedido mas reciente en prior)
last_order_num = orders_prior.groupby('user_id')['order_number'].max()

# Usuarios con al menos 2 pedidos prior (necesitamos al menos 1 de train)
eligible_users = last_order_num[last_order_num >= 2].index
print(f'Usuarios elegibles para evaluacion (>=2 pedidos prior): {len(eligible_users):,}')

orders_prior_eligible = orders_prior[orders_prior['user_id'].isin(eligible_users)]
last_orders = orders_prior_eligible.sort_values('order_number').groupby('user_id').last().reset_index()
test_order_ids = set(last_orders['order_id'])

op_user_eligible = op_user[op_user['user_id'].isin(eligible_users)].copy()
train_op = op_user_eligible[~op_user_eligible['order_id'].isin(test_order_ids)]
test_op  = op_user_eligible[ op_user_eligible['order_id'].isin(test_order_ids)]

print(f'Transacciones de train : {len(train_op):,}')
print(f'Transacciones de test  : {len(test_op):,}')

# Ground truth: productos del ultimo pedido por usuario
truth_dict = test_op.groupby('user_id')['product_id'].apply(set).to_dict()
print(f'Usuarios de test con ground truth: {len(truth_dict):,}')


Usuarios elegibles para evaluacion (>=2 pedidos prior): 206,209
Transacciones de train : 30,294,701
Transacciones de test  : 2,139,788
Usuarios de test con ground truth: 206,209


## 3. Filtrar productos por soporte mínimo de usuarios

El catálogo tiene ~49.700 productos. Si intentáramos calcular la similitud coseno entre **todos contra todos**, la matriz de similitud tendría ~49.700² ≈ 2.470 millones de celdas — inviable en memoria y, peor, sin sentido estadístico: un producto que compraron 3 personas en toda la historia no tiene señal suficiente para estimar con quién "se parece".

Aplicamos el mismo criterio que usamos en el MBA (`MIN_SUPPORT`), adaptado a este contexto: en vez de medir en cuántos **pedidos** aparece un producto, medimos en cuántos **usuarios distintos** lo compraron alguna vez (porque acá la matriz es usuario-producto, no pedido-producto).

```
soporte_usuario(producto) = (usuarios que compraron el producto al menos una vez) / (usuarios totales)
```

Calculamos esto **solo sobre train** (para no filtrar usando información de test) y probamos varios umbrales para elegir uno que deje un catálogo manejable sin perder productos relevantes.


In [50]:
n_users_train = train_op['user_id'].nunique()
print(f'Usuarios en train: {n_users_train:,}')

# soporte de usuario = cuantos usuarios distintos compraron el producto (en train), sobre el total
user_reach = (
    train_op.drop_duplicates(['user_id', 'product_id'])
            .groupby('product_id').size() / n_users_train
)

print('\nCuantos productos sobreviven segun el umbral elegido:')
for thr in [0.001, 0.002, 0.003, 0.005, 0.01]:
    print(f'  MIN_USER_SUPPORT={thr:.3f}  ->  {(user_reach >= thr).sum():,} productos')

print(f'\nUsamos MIN_USER_SUPPORT = {MIN_USER_SUPPORT} (deja un catalogo manejable, en el orden de miles de productos,')
print('cubriendo los productos con senal de compra suficiente para estimar similitud de forma confiable).')

filtered_products = user_reach[user_reach >= MIN_USER_SUPPORT].index
filtered_products = np.sort(filtered_products.values)
n_items = len(filtered_products)
print(f'\nProductos en la matriz de similitud: {n_items:,} de {products["product_id"].nunique():,} totales')


Usuarios en train: 206,209

Cuantos productos sobreviven segun el umbral elegido:
  MIN_USER_SUPPORT=0.001  ->  9,075 productos
  MIN_USER_SUPPORT=0.002  ->  5,443 productos
  MIN_USER_SUPPORT=0.003  ->  3,788 productos
  MIN_USER_SUPPORT=0.005  ->  2,295 productos
  MIN_USER_SUPPORT=0.010  ->  1,053 productos

Usamos MIN_USER_SUPPORT = 0.003 (deja un catalogo manejable, en el orden de miles de productos,
cubriendo los productos con senal de compra suficiente para estimar similitud de forma confiable).

Productos en la matriz de similitud: 3,788 de 49,688 totales


## 4. Construir la matriz usuario-producto (binaria, dispersa)

Con ~200.000 usuarios x ~4.000 productos, la matriz tiene ~800 millones de celdas — pero la gran mayoría son `0` (ningún usuario compra ni el 1% del catálogo). Guardar eso como un array denso de numpy sería un desperdicio enorme de memoria.

Usamos una **matriz dispersa** (*sparse matrix*, `scipy.sparse.csr_matrix`): en vez de guardar todas las celdas, guarda solo las posiciones donde hay un `1`. Es exactamente la misma idea que usamos para la matriz "canasta" en el MBA (NB07/08).


In [51]:
# Restringimos train_op a los productos filtrados
train_op_f = train_op[train_op['product_id'].isin(filtered_products)]

# Mapeos id <-> indice de matriz
user_ids_sorted = np.sort(train_op_f['user_id'].unique())
user_to_idx = {u: i for i, u in enumerate(user_ids_sorted)}
item_to_idx = {p: i for i, p in enumerate(filtered_products)}
idx_to_item = {i: p for p, i in item_to_idx.items()}

rows = train_op_f['user_id'].map(user_to_idx).values
cols = train_op_f['product_id'].map(item_to_idx).values
data = np.ones(len(train_op_f), dtype='int8')

train_matrix = csr_matrix(
    (data, (rows, cols)),
    shape=(len(user_ids_sorted), n_items),
)
# Por si un usuario compro el mismo producto en varios pedidos de train: lo binarizamos
train_matrix.data[:] = 1

density = train_matrix.nnz / (train_matrix.shape[0] * train_matrix.shape[1])
print(f'train_matrix shape : {train_matrix.shape}  (usuarios x productos filtrados)')
print(f'nnz (compras no-cero): {train_matrix.nnz:,}')
print(f'densidad: {density:.5f}  ({density*100:.3f}% de las celdas tienen un 1)')


train_matrix shape : (205733, 3788)  (usuarios x productos filtrados)
nnz (compras no-cero): 8,842,237
densidad: 0.01135  (1.135% de las celdas tienen un 1)


## 5. Calcular la similitud coseno item-item

Para comparar **productos** en vez de usuarios, trasponemos la matriz (`train_matrix.T`): ahora cada fila es un producto, descrito por el vector de usuarios que lo compraron. `cosine_similarity` de scikit-learn calcula la similitud coseno entre **todas las filas contra todas las filas**, dando como resultado una matriz cuadrada `productos x productos`.

La diagonal de esa matriz es la similitud de cada producto consigo mismo, que siempre da `1` — la ponemos en `0` explícitamente para que nunca se recomiende un producto a sí mismo.


In [52]:
item_vectors = train_matrix.T  # ahora: productos x usuarios

similarity_matrix = cosine_similarity(item_vectors)  # denso, n_items x n_items
similarity_matrix = similarity_matrix.astype('float32')
np.fill_diagonal(similarity_matrix, 0.0)  # un producto no es "similar a si mismo" para efectos de recomendar

mem_mb = similarity_matrix.nbytes / (1024 ** 2)
print(f'similarity_matrix shape: {similarity_matrix.shape}')
print(f'memoria: {mem_mb:.1f} MB')
print(f'similitud minima: {similarity_matrix.min():.4f}  maxima: {similarity_matrix.max():.4f}')
print(f'similitud promedio (excluyendo ceros exactos): {similarity_matrix[similarity_matrix > 0].mean():.4f}')


similarity_matrix shape: (3788, 3788)
memoria: 54.7 MB
similitud minima: 0.0000  maxima: 0.6889
similitud promedio (excluyendo ceros exactos): 0.0187


## 6. Explorar vecinos más cercanos de un producto (chequeo cualitativo)

Antes de armar la función de recomendación completa, miremos directamente: para un producto dado, ¿cuáles son sus vecinos más cercanos según el coseno? Esto es un primer chequeo de "sentido común" — si el modelo dice que la leche de almendras es súper similar a las pilas AA, algo está mal.


In [53]:
def get_similar_items(product_id, top_n=10):
    '''Devuelve los top_n productos mas similares (coseno) a product_id.

    Parameters
    ----------
    product_id : int
    top_n : int

    Returns
    -------
    pd.DataFrame con columnas [product_id, product_name, similarity]
    '''
    if product_id not in item_to_idx:
        raise ValueError(
            f'product_id {product_id} no esta en el catalogo filtrado '
            f'(soporte de usuario < {MIN_USER_SUPPORT}). No tiene vecinos calculados.'
        )
    idx = item_to_idx[product_id]
    sims = similarity_matrix[idx]
    top_idx = np.argsort(sims)[::-1][:top_n]

    return pd.DataFrame({
        'product_id':  [idx_to_item[i] for i in top_idx],
        'product_name':[id_to_name[idx_to_item[i]] for i in top_idx],
        'similarity':  sims[top_idx],
    })


# Probemos con un par de productos conocidos
for example_pid in [21903, 5876]:  # Organic Baby Spinach, Bag of Organic Bananas
    print(f'Vecinos mas similares a {example_pid} ({id_to_name[example_pid]}):')
    display(get_similar_items(example_pid, top_n=8))
    print()


Vecinos mas similares a 21903 (Organic Baby Spinach):


,product_id,product_name,similarity
0,21137,Organic Strawberries,0.475530
1,24964,Organic Garlic,0.458573
2,47209,Organic Hass Avocado,0.442198
3,13176,Bag of Organic Bananas,0.436787
4,45007,Organic Zucchini,0.425018
5,47766,Organic Avocado,0.423802
6,22935,Organic Yellow Onion,0.419600
7,26209,Limes,0.410737



Vecinos mas similares a 5876 (Organic Lemon):


,product_id,product_name,similarity
0,47209,Organic Hass Avocado,0.431088
1,22935,Organic Yellow Onion,0.394428
2,24964,Organic Garlic,0.390789
3,21137,Organic Strawberries,0.368621
4,13176,Bag of Organic Bananas,0.360761
5,21903,Organic Baby Spinach,0.349560
6,26209,Limes,0.345589
7,34126,Organic Italian Parsley Bunch,0.343248


## 7. Política de recomendación: no recomendar lo obvio

Igual que en el MBA (sección 7.5 de NB07), un producto extremadamente popular (banana) va a tener *algo* de similitud con casi cualquier otro producto, simplemente porque lo compra casi todo el mundo. El coseno ya **atenúa** este efecto mucho más que una cuenta cruda de co-ocurrencias (porque divide por la popularidad de ambos productos) — pero no lo elimina del todo.

Aplicamos el mismo tipo de filtro: no recomendar productos cuyo soporte de usuario (popularidad) supere `MAX_CONSEQUENT_SUPPORT`. El umbral es más alto que en el MBA (`0.20` vs `0.10`) porque el coseno ya hace buena parte del trabajo de-sesgando; este filtro es una segunda capa de seguridad, no la principal.


In [54]:
product_support = user_reach  # ya lo calculamos en la seccion 3, sobre train

blocked = product_support[(product_support > MAX_CONSEQUENT_SUPPORT) & (product_support.index.isin(filtered_products))]
blocked = blocked.sort_values(ascending=False)
print(f'Con MAX_CONSEQUENT_SUPPORT={MAX_CONSEQUENT_SUPPORT}, se excluyen {len(blocked)} productos como recomendacion:')
for pid, supp in blocked.items():
    print(f'  {pid:>6}  {supp*100:5.1f}%  {id_to_name[pid]}')


Con MAX_CONSEQUENT_SUPPORT=0.3, se excluyen 1 productos como recomendacion:
   24852   34.1%  Banana


## 8. Función recomendadora: de un carrito a una lista de sugerencias

La lógica, para un carrito con varios productos:

1. Para cada producto candidato (que no esté ya en el carrito), sumamos su similitud coseno con **cada** producto del carrito. Esto es el mismo principio que "cuántos vecinos del carrito apuntan a este candidato, y con qué fuerza".
2. Excluimos productos ya en el carrito.
3. Aplicamos la política de la sección 7 (no recomendar lo obvio).
4. Ordenamos de mayor a menor score agregado.

Si algún producto del carrito no está en el catálogo filtrado (muy poco comprado), simplemente no aporta señal — lo avisamos pero seguimos con el resto.


In [55]:
def recommend_from_cart(cart_product_ids, top_n=10, max_consequent_support=MAX_CONSEQUENT_SUPPORT):
    '''Recomienda productos a partir de un carrito, usando similitud coseno item-item.

    Parameters
    ----------
    cart_product_ids : list[int]
    top_n : int
    max_consequent_support : float o None
        no se recomienda ningun producto cuyo soporte de usuario supere este valor.
        Pasa None para desactivar el filtro.

    Returns
    -------
    pd.DataFrame con columnas [product_id, product_name, score]
    '''
    cart_idx = [item_to_idx[p] for p in cart_product_ids if p in item_to_idx]
    missing  = [p for p in cart_product_ids if p not in item_to_idx]
    if missing:
        print(f'Aviso: {missing} no estan en el catalogo filtrado (soporte insuficiente), se ignoran para el score.')
    if not cart_idx:
        return pd.DataFrame(columns=['product_id', 'product_name', 'score'])

    # score agregado = suma de similitudes del candidato con cada producto del carrito
    scores = similarity_matrix[cart_idx, :].sum(axis=0)

    cart_set = set(cart_product_ids)
    result = pd.DataFrame({
        'product_id': [idx_to_item[i] for i in range(n_items)],
        'score':      scores,
    })
    result = result[~result['product_id'].isin(cart_set)]

    if max_consequent_support is not None:
        support_of = result['product_id'].map(product_support)
        result = result[support_of <= max_consequent_support]

    result = result[result['score'] > 0]
    result['product_name'] = result['product_id'].map(id_to_name)
    result = result.sort_values('score', ascending=False).head(top_n).reset_index(drop=True)
    return result[['product_id', 'product_name', 'score']]


## 9. Prueba manual: mismo carrito que usamos en el MBA

Usamos el **mismo carrito** que en NB07/NB08 (`Organic Baby Spinach` + `Organic Hass Avocado`) para poder comparar directamente qué recomienda cada enfoque frente al mismo input.


In [56]:
cart = [20114, 30391]  # Jalapeno Peppers, Organic Cucumber
print('Carrito actual:')
for pid in cart:
    print(f'  - {pid}: {id_to_name[pid]}')

print()
print('Recomendaciones SIN de-sesgar (max_consequent_support=None):')
display(recommend_from_cart(cart, max_consequent_support=None))

print()
print(f'Recomendaciones CON de-sesgar (max_consequent_support={MAX_CONSEQUENT_SUPPORT}, el default):')
recommend_from_cart(cart)


Carrito actual:
  - 20114: Jalapeno Peppers
  - 30391: Organic Cucumber

Recomendaciones SIN de-sesgar (max_consequent_support=None):


,product_id,product_name,score
0,26209,Limes,0.671363
1,24964,Organic Garlic,0.623184
2,22935,Organic Yellow Onion,0.611270
3,47209,Organic Hass Avocado,0.608818
4,21903,Organic Baby Spinach,0.605170
5,21137,Organic Strawberries,0.598911
6,31717,Organic Cilantro,0.593924
7,45007,Organic Zucchini,0.586604
8,47626,Large Lemon,0.581492
9,8518,Organic Red Onion,0.580363



Recomendaciones CON de-sesgar (max_consequent_support=0.3, el default):


,product_id,product_name,score
0,26209,Limes,0.671363
1,24964,Organic Garlic,0.623184
2,22935,Organic Yellow Onion,0.611270
3,47209,Organic Hass Avocado,0.608818
4,21903,Organic Baby Spinach,0.605170
5,21137,Organic Strawberries,0.598911
6,31717,Organic Cilantro,0.593924
7,45007,Organic Zucchini,0.586604
8,47626,Large Lemon,0.581492
9,8518,Organic Red Onion,0.580363


Compará esta tabla con la de `recommend_from_cart` en NB07 (`docs`/notebook de MBA) para el mismo carrito: es esperable que la lista **no sea idéntica** — el MBA busca productos que aparecen *en el mismo pedido*, mientras que este modelo busca productos comprados *por los mismos usuarios* a lo largo de toda su historia. Ambas señales son válidas y capturan cosas distintas; por eso en un sistema real suele convenir combinarlas (o al menos mostrar ambas).


In [57]:
# Probemos un carrito con un producto de bajo soporte, para ver el caso "sin señal"
# Buscamos algun producto raro en el catalogo para armar el ejemplo
raro = products[~products['product_id'].isin(filtered_products)].sample(1, random_state=RANDOM_STATE)
raro_pid = int(raro['product_id'].iloc[0])
print(f'Producto de bajo soporte elegido: {raro_pid} - {id_to_name[raro_pid]}')

cart_2 = [raro_pid]
print('\nRecomendaciones para un carrito con solo ese producto:')
recommend_from_cart(cart_2)


Producto de bajo soporte elegido: 5837 - Collard Mustard Turnip Greens Southern Blend

Recomendaciones para un carrito con solo ese producto:
Aviso: [5837] no estan en el catalogo filtrado (soporte insuficiente), se ignoran para el score.


,product_id,product_name,score


## Fase 3 — Validación cuantitativa (A): métricas de tarea

Ahora medimos qué tan bien el modelo predice el **próximo pedido real** de cada usuario de test, usando el mismo esquema y las mismas métricas que en NB06 — así los números son directamente comparables entre notebooks.

**La "consulta" para cada usuario es su propio historial de compras en train** (en vez de un carrito de 2 productos como en la demo de arriba). La idea: "dado todo lo que este usuario compró antes, ¿qué le recomendamos para su próximo pedido?" — y comparamos contra lo que realmente compró.

**Por qué no hace falta iterar usuario por usuario:** el score de cada candidato para un usuario es la suma de similitudes con todo lo que el usuario compró en train. Eso es exactamente una **multiplicación de matrices**: `train_matrix (usuarios x productos) @ similarity_matrix (productos x productos) = scores (usuarios x productos)`. Con `scipy` esto corre en segundos para los ~200.000 usuarios, en lotes (*batches*) para no acumular toda la matriz de scores en memoria de una sola vez.


In [58]:
BATCH_SIZE = 10_000
n_users_eval = train_matrix.shape[0]
max_k = max(K_VALUES)

# vector de soporte alineado al orden de columnas de la matriz de similitud (indice 0..n_items-1)
support_vec = np.array([product_support.get(idx_to_item[i], 0.0) for i in range(n_items)], dtype='float32')
blocked_mask = support_vec > MAX_CONSEQUENT_SUPPORT  # True = no se puede recomendar (politica seccion 7)

recs_dict = {}
idx_to_user = {i: u for u, i in user_to_idx.items()}

for start in range(0, n_users_eval, BATCH_SIZE):
    end = min(start + BATCH_SIZE, n_users_eval)
    batch = train_matrix[start:end]                     # (batch_size, n_items) sparse
    scores_batch = batch.dot(similarity_matrix)          # (batch_size, n_items) denso

    # no recomendar lo que el usuario ya compro (en train), ni lo bloqueado por la politica
    already_bought = batch.toarray().astype(bool)
    scores_batch[already_bought] = -np.inf
    scores_batch[:, blocked_mask] = -np.inf

    top_idx_batch = np.argpartition(-scores_batch, max_k, axis=1)[:, :max_k]

    for local_i in range(end - start):
        user_id = idx_to_user[start + local_i]
        cand_idx = top_idx_batch[local_i]
        cand_scores = scores_batch[local_i, cand_idx]
        order = np.argsort(-cand_scores)
        ranked_idx = cand_idx[order]
        ranked_idx = ranked_idx[np.isfinite(cand_scores[order])]  # descartamos -inf (sin señal real)
        recs_dict[user_id] = [idx_to_item[i] for i in ranked_idx]

print(f'Recomendaciones generadas para {len(recs_dict):,} usuarios de train.')
print(f'Ejemplo (usuario {next(iter(recs_dict))}): {recs_dict[next(iter(recs_dict))][:5]}')


Recomendaciones generadas para 205,733 usuarios de train.
Ejemplo (usuario 1): [np.int32(6184), np.int32(21137), np.int32(37710), np.int32(38928), np.int32(16797)]


### Función `evaluate_recommendations`

Misma función y misma firma que en NB06 (contrato inter-modelos), para que los números sean comparables.

| Métrica | Descripción |
|---|---|
| `precision@k` | De las top-k recomendaciones, qué fracción estaban en el ground truth (el usuario realmente las compró en su próximo pedido) |
| `recall@k` | Del ground truth (todo lo que compró en su próximo pedido), qué fracción logramos incluir en las top-k |
| `hit_rate@k` | Fracción de usuarios con al menos 1 acierto en las top-k (métrica "todo o nada", más fácil de interpretar) |
| `MAP@k` | *Mean Average Precision*; premia que los aciertos aparezcan temprano en la lista (no solo que estén) |


In [59]:
def evaluate_recommendations(recs_dict, truth_dict, k_values=None):
    '''
    Evalua recomendaciones con metricas de ranking.
    Misma firma que en NB06 (contrato inter-modelos).

    Parameters
    ----------
    recs_dict  : dict {user_id: list[product_id]}  lista ordenada de recomendaciones
    truth_dict : dict {user_id: set(product_id)}   ground truth (ultimo pedido)
    k_values   : list[int]

    Returns
    -------
    dict {k: {precision, recall, hit_rate, map, n_users}}
    '''
    if k_values is None:
        k_values = [5, 10, 20]

    users = [u for u in recs_dict if u in truth_dict and truth_dict[u]]
    results = {}

    for k in k_values:
        prec_list, rec_list, hit_list, ap_list = [], [], [], []
        for u in users:
            recs     = recs_dict[u][:k]
            truth    = truth_dict[u]
            hits_set = set(recs) & truth

            prec_list.append(len(hits_set) / k)
            rec_list.append(len(hits_set) / len(truth))
            hit_list.append(1 if hits_set else 0)

            ap, n_hits = 0.0, 0
            for i, pid in enumerate(recs, 1):
                if pid in truth:
                    n_hits += 1
                    ap += n_hits / i
            ap /= min(len(truth), k)
            ap_list.append(ap)

        results[k] = {
            'precision': float(np.mean(prec_list)),
            'recall':    float(np.mean(rec_list)),
            'hit_rate':  float(np.mean(hit_list)),
            'map':       float(np.mean(ap_list)),
            'n_users':   len(prec_list),
        }

    return results


print('Calculando metricas...')
metrics = evaluate_recommendations(recs_dict, truth_dict, k_values=K_VALUES)

print('\nMetricas de evaluacion:')
for k, m in metrics.items():
    f1 = 2 * m['precision'] * m['recall'] / (m['precision'] + m['recall'] + 1e-10)
    print(f'  k={k:2d}  Precision={m["precision"]:.4f}  Recall={m["recall"]:.4f}  '
          f'F1={f1:.4f}  HitRate={m["hit_rate"]:.4f}  MAP={m["map"]:.4f}  (n={m["n_users"]:,})')


Calculando metricas...

Metricas de evaluacion:
  k= 5  Precision=0.0242  Recall=0.0124  F1=0.0164  HitRate=0.1072  MAP=0.0140  (n=205,733)
  k=10  Precision=0.0203  Recall=0.0204  F1=0.0203  HitRate=0.1623  MAP=0.0104  (n=205,733)
  k=20  Precision=0.0161  Recall=0.0319  F1=0.0214  HitRate=0.2293  MAP=0.0095  (n=205,733)


### Cobertura del catálogo (*Coverage*)

Igual que en NB06: la cobertura mide qué fracción del catálogo total llega a recomendarse **a alguien**. A diferencia de Popularidad (que siempre recomienda lo mismo, cobertura mínima), un modelo personalizado como este debería recomendar un conjunto **mucho más amplio** de productos distintos, porque cada usuario tiene una historia de compra diferente.


In [60]:
n_catalog = products['product_id'].nunique()
all_recommended = set()
for recs in recs_dict.values():
    all_recommended.update(recs[:max(K_VALUES)])

for k in K_VALUES:
    recommended_at_k = set()
    for recs in recs_dict.values():
        recommended_at_k.update(recs[:k])
    cov = len(recommended_at_k) / n_catalog
    print(f'  Coverage@{k:2d}: {cov:.4f}  ({len(recommended_at_k):,} de {n_catalog:,} productos distintos recomendados)')

print(f'\n(Esperado: mucho mayor que Popularidad, que solo cubre {max(K_VALUES)} productos en total)')


  Coverage@ 5: 0.0328  (1,629 de 49,688 productos distintos recomendados)
  Coverage@10: 0.0390  (1,937 de 49,688 productos distintos recomendados)
  Coverage@20: 0.0465  (2,310 de 49,688 productos distintos recomendados)

(Esperado: mucho mayor que Popularidad, que solo cubre 20 productos en total)


## Fase 3 — Validación (B): coherencia de categoría (chequeo intrínseco)

Las métricas de arriba miden si el modelo predice bien el **próximo pedido** — pero eso depende también de cuánto varían las compras de la gente (nadie predice perfecto qué vas a comprar). Acá hacemos una pregunta distinta, específica de un modelo de similitud: **¿las relaciones que aprendió el modelo tienen sentido de categoría de producto?**

La idea: para cada producto, miramos sus `k` vecinos más cercanos (según el coseno) y calculamos qué fracción cae en el **mismo pasillo** (`aisle_id`). Si el modelo aprendió señal real, los vecinos de "leche de almendras" deberían ser mayormente otras leches/bebidas vegetales, no productos al azar.

**Comparamos contra un baseline aleatorio:** si eligiéramos vecinos al azar del catálogo filtrado, ¿qué fracción caería en el mismo pasillo solo por casualidad? Eso depende de cuántos productos hay en cada pasillo (pasillos grandes tienen más chance de "acertar" al azar). La diferencia entre el resultado real y ese baseline es la señal que aporta el modelo.


In [61]:
TOP_K_COHERENCE = 10

# aisle_id de cada producto filtrado, alineado al indice de la matriz
item_aisles = np.array([product_to_aisle[idx_to_item[i]] for i in range(n_items)])

# baseline aleatorio: prob. de que 2 productos del catalogo filtrado compartan pasillo,
# elegidos al azar (sin reemplazo, aproximado)
aisle_counts = pd.Series(item_aisles).value_counts()
p_random = ((aisle_counts * (aisle_counts - 1)).sum()) / (n_items * (n_items - 1))

same_aisle_fracs = []
rng = np.random.default_rng(RANDOM_STATE)
sample_items = rng.choice(n_items, size=min(1000, n_items), replace=False)  # muestra para que corra rapido

for idx in sample_items:
    sims = similarity_matrix[idx].copy()
    top_idx = np.argsort(sims)[::-1][:TOP_K_COHERENCE]
    same_aisle = (item_aisles[top_idx] == item_aisles[idx]).mean()
    same_aisle_fracs.append(same_aisle)

coherence = float(np.mean(same_aisle_fracs))

print(f'Coherencia de categoria (fraccion de vecinos en el mismo pasillo), top-{TOP_K_COHERENCE}:')
print(f'  Modelo (real)     : {coherence:.4f}')
print(f'  Baseline aleatorio: {p_random:.4f}')
print(f'  Ratio modelo/azar : {coherence / p_random:.1f}x')
print(f'\nInterpretacion: si el ratio es >> 1, el modelo esta agrupando productos de forma')
print('mucho mas coherente que el azar -- señal de que el coseno esta capturando relaciones reales,')
print('no ruido. Un ratio cercano a 1 indicaria que las similitudes no aportan informacion util.')


Coherencia de categoria (fraccion de vecinos en el mismo pasillo), top-10:
  Modelo (real)     : 0.2979
  Baseline aleatorio: 0.0191
  Ratio modelo/azar : 15.6x

Interpretacion: si el ratio es >> 1, el modelo esta agrupando productos de forma
mucho mas coherente que el azar -- señal de que el coseno esta capturando relaciones reales,
no ruido. Un ratio cercano a 1 indicaria que las similitudes no aportan informacion util.


## Tabla resumen de métricas

In [62]:
rows = []
for k, m in metrics.items():
    f1 = 2 * m['precision'] * m['recall'] / (m['precision'] + m['recall'] + 1e-10)
    recommended_at_k = set()
    for recs in recs_dict.values():
        recommended_at_k.update(recs[:k])
    rows.append({
        'k': k,
        'precision': round(m['precision'], 4),
        'recall': round(m['recall'], 4),
        'f1': round(f1, 4),
        'hit_rate': round(m['hit_rate'], 4),
        'map': round(m['map'], 4),
        'coverage': round(len(recommended_at_k) / n_catalog, 4),
    })

summary_df = pd.DataFrame(rows)
summary_df.attrs['category_coherence_ratio'] = round(coherence / p_random, 2)
print(f'Coherencia de categoria (ratio vs azar): {coherence / p_random:.2f}x\n')
display(summary_df)


Coherencia de categoria (ratio vs azar): 15.64x



,k,precision,recall,f1,hit_rate,map,coverage
0,5,0.0242,0.0124,0.0164,0.1072,0.0140,0.0328
1,10,0.0203,0.0204,0.0203,0.1623,0.0104,0.0390
2,20,0.0161,0.0319,0.0214,0.2293,0.0095,0.0465


## Fase 4 — Reentrenar sobre todo `prior` y empaquetar modelo

Igual que en NB06/07/08: para producción reentrenamos usando **todos** los pedidos `prior` (no solo train), para aprovechar toda la señal disponible. Las métricas de arriba quedan calculadas con el modelo de train (sin fuga de datos); el modelo que exportamos es el de producción.


In [63]:
# Repetimos filtro + matriz + similitud, esta vez sobre TODO prior
n_users_full = op_user['user_id'].nunique()
user_reach_full = (
    op_user.drop_duplicates(['user_id', 'product_id'])
           .groupby('product_id').size() / n_users_full
)
filtered_products_full = np.sort(user_reach_full[user_reach_full >= MIN_USER_SUPPORT].index.values)
n_items_full = len(filtered_products_full)
print(f'Productos en catalogo de produccion: {n_items_full:,}')

op_full_f = op_user[op_user['product_id'].isin(filtered_products_full)]
user_ids_full = np.sort(op_full_f['user_id'].unique())
user_to_idx_full = {u: i for i, u in enumerate(user_ids_full)}
item_to_idx_full = {p: i for i, p in enumerate(filtered_products_full)}
idx_to_item_full = {i: p for p, i in item_to_idx_full.items()}

rows_f = op_full_f['user_id'].map(user_to_idx_full).values
cols_f = op_full_f['product_id'].map(item_to_idx_full).values
data_f = np.ones(len(op_full_f), dtype='int8')

full_matrix = csr_matrix((data_f, (rows_f, cols_f)), shape=(len(user_ids_full), n_items_full))
full_matrix.data[:] = 1

full_similarity = cosine_similarity(full_matrix.T).astype('float32')
np.fill_diagonal(full_similarity, 0.0)

product_support_full = user_reach_full

print(f'full_matrix shape     : {full_matrix.shape}')
print(f'full_similarity shape : {full_similarity.shape}  ({full_similarity.nbytes / (1024**2):.1f} MB)')


Productos en catalogo de produccion: 4,045
full_matrix shape     : (205957, 4045)
full_similarity shape : (4045, 4045)  (62.4 MB)


### Exportar artefactos

Empaquetamos todo lo necesario para recomendar (matriz de similitud, mapeos id<->indice, soporte de usuario, nombres de producto) en un único artefacto `.joblib` autocontenido, igual que en `src/recommender.py` y `src/aisle_recommender.py`. También guardamos las métricas de evaluación y el catálogo compartido.


In [64]:
def _to_py(obj):
    '''Convierte tipos numpy a tipos nativos de Python para poder serializar a JSON.'''
    if isinstance(obj, dict):
        return {str(k): _to_py(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_to_py(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    return obj


MODEL_PATH   = MODELS / 'cf_item_item_model.joblib'
METRICS_PATH = DATA_PROC / 'cf_item_item_metrics.json'
CATALOG_PATH = DATA_PROC / 'products_catalog.csv'

artifact = {
    'similarity_matrix': full_similarity,
    'item_to_idx': item_to_idx_full,
    'idx_to_item': idx_to_item_full,
    'product_support': product_support_full.to_dict(),
    'id_to_name': id_to_name,
    'max_consequent_support': MAX_CONSEQUENT_SUPPORT,
    'metadata': {
        'generated_at': datetime.now(timezone.utc).isoformat(),
        'min_user_support': MIN_USER_SUPPORT,
        'n_items': n_items_full,
        'n_users_train_full': len(user_ids_full),
        'source_notebook': '09_Item_Item_Cosine.ipynb',
    },
}
joblib.dump(artifact, MODEL_PATH)
print(f'modelo exportado a {MODEL_PATH}')

metrics_export = {
    'k_values': K_VALUES,
    'metrics_by_k': _to_py(metrics),
    'category_coherence_ratio_vs_random': round(coherence / p_random, 4),
    'min_user_support': MIN_USER_SUPPORT,
    'max_consequent_support': MAX_CONSEQUENT_SUPPORT,
    'n_items_eval': n_items,
}
with open(METRICS_PATH, 'w', encoding='utf-8') as f:
    json.dump(metrics_export, f, indent=2, ensure_ascii=False)
print(f'metricas exportadas a {METRICS_PATH}')

products[['product_id', 'product_name']].to_csv(CATALOG_PATH, index=False)
print(f'catalogo exportado a {CATALOG_PATH}')


modelo exportado a ..\models\cf_item_item_model.joblib
metricas exportadas a ..\data\processed\cf_item_item_metrics.json
catalogo exportado a ..\data\processed\products_catalog.csv


## Código productivo en `src/cf_item_item_recommender.py`

Igual que con el MBA: la función `recommend_from_cart` de arriba queda en la notebook por su valor didáctico (para leerla paso a paso), y `src/cf_item_item_recommender.py` es la versión que se importa desde otro código. Misma lógica, misma firma de salida que `Recommender.recommend` (`src/recommender.py`): `[{'product_id', 'product_name', 'rank'}, ...]`.


In [65]:
import sys
sys.path.append('../src')
from cf_item_item_recommender import CFItemItemRecommender

model = CFItemItemRecommender.load()  # sin argumentos: encuentra models/cf_item_item_model.joblib solo

for test_cart in [cart, [13176]]:
    ids_notebook = recommend_from_cart(test_cart)['product_id'].tolist() if all(p in item_to_idx for p in test_cart) else None
    ids_module   = [r['product_id'] for r in model.recommend(test_cart)]

    print('carrito:', [id_to_name[p] for p in test_cart])
    print('  CFItemItemRecommender.recommend:', ids_module)
    print()

print('salida final (formato exportable):')
model.recommend(cart)


carrito: ['Jalapeno Peppers', 'Organic Cucumber']
  CFItemItemRecommender.recommend: [np.int32(26209), np.int32(24964), np.int32(21903), np.int32(22935), np.int32(47209), np.int32(21137), np.int32(31717), np.int32(47626), np.int32(45007), np.int32(8518)]

carrito: ['Bag of Organic Bananas']
  CFItemItemRecommender.recommend: [np.int32(21137), np.int32(47209), np.int32(21903), np.int32(27966), np.int32(39275), np.int32(22935), np.int32(24964), np.int32(5876), np.int32(45007), np.int32(30391)]

salida final (formato exportable):


[{'product_id': np.int32(26209), 'product_name': 'Limes', 'rank': 1},
 {'product_id': np.int32(24964), 'product_name': 'Organic Garlic', 'rank': 2},
 {'product_id': np.int32(21903),
  'product_name': 'Organic Baby Spinach',
  'rank': 3},
 {'product_id': np.int32(22935),
  'product_name': 'Organic Yellow Onion',
  'rank': 4},
 {'product_id': np.int32(47209),
  'product_name': 'Organic Hass Avocado',
  'rank': 5},
 {'product_id': np.int32(21137),
  'product_name': 'Organic Strawberries',
  'rank': 6},
 {'product_id': np.int32(31717),
  'product_name': 'Organic Cilantro',
  'rank': 7},
 {'product_id': np.int32(47626), 'product_name': 'Large Lemon', 'rank': 8},
 {'product_id': np.int32(45007),
  'product_name': 'Organic Zucchini',
  'rank': 9},
 {'product_id': np.int32(8518),
  'product_name': 'Organic Red Onion',
  'rank': 10}]

## Conclusiones

- El recomendador item-item con similitud coseno captura una señal **distinta y complementaria** al Market Basket Analysis: relaciones "estas dos cosas las compra el mismo tipo de usuario a lo largo del tiempo", en vez de "estas dos cosas van juntas en el mismo pedido".
- La cobertura del catálogo es sustancialmente mayor que la de Popularidad, porque las recomendaciones dependen del historial de cada usuario.
- La coherencia de categoría (chequeo B) confirma que las similitudes aprendidas no son ruido: los vecinos más cercanos de un producto caen, con frecuencia muy superior al azar, en su mismo pasillo.
- Las métricas de tarea (Precision/Recall/HitRate/MAP@k, chequeo A) son comparables directamente contra NB06 (piso) y contra el MBA, usando el mismo split *leave-last-order-out*.
- Quedan disponibles dos artefactos reutilizables: `models/cf_item_item_model.joblib` (el modelo) y `src/cf_item_item_recommender.py` (la clase `CFItemItemRecommender` para consumirlo desde cualquier script).
